# FAIR CARE SURVEY Data More Semantic Groupings 


In [1]:
from collections import Counter

import nltk
from nltk import ngrams
from nltk.util import everygrams
from nltk.corpus import stopwords

import os

import pandas as pd

import re
import statistics

# Get the root_path for this jupyter notebook repo.
repo_path = os.path.dirname(os.path.abspath(os.getcwd()))
# This is the path to a CSV file that provides configurations and metadata about the survey columns.
col_config_path = os.path.join(
    repo_path, 'files', 'IMLS-FAIR-CARE-Survey', 'imls-fair-care-survey-columns-config.csv',
)

processed_survey_path = '/home/ekansa/oc-data/fair-care-survey-processed.csv' # Keep this OUT of version control, has sensitive info
semantic_clusters_path = '/home/ekansa/oc-data/fair-care-semantic-clusters.csv'
exp_semantic_clusters_path = '/home/ekansa/oc-data/fair-care-semantic-clusters-expanded.csv'

df_config = pd.read_csv(col_config_path, low_memory=False)
df = pd.read_csv(processed_survey_path, low_memory=False)
df_semantic = pd.read_csv(semantic_clusters_path, low_memory=False)


## Step 1: Add a word count column to help describe each text response

A count of words for each response may indicate the level of interest in a given question.

In [2]:

nltk.download('stopwords')
nltk.download('punkt_tab')

if not 'word_count' in df_semantic.columns.tolist():
    # Add a word_count if it is not already present.
    df_semantic['resp_text'] = df_semantic['response'].str.lower().str.replace('[^\w\s]','')
    df_semantic['word_count'] = df_semantic['resp_text'].apply(lambda n: len(n.split()))
    df_semantic.drop(columns=['resp_text'], inplace=True)


[nltk_data] Downloading package stopwords to /home/ekansa/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ekansa/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Step 2: Add data that combines the full-question and reponse answer

Let's test to see if we get more meaningful semantic clusters by combining the question and the anwser together into the same text for generating embeddings.

In [3]:
def get_original_column_name(orig_column_index, df_config=df_config):
    """Gets the original column name from the df_config (column configuration data)"""
    config_index = (df_config['orig_column_index'] == orig_column_index)
    if len(df_config[config_index].index) != 1:
        # We didn't find a matching column name
        return None
    row = df_config[config_index].iloc[0]
    return row['raw_column']

df_semantic['q_a_txt'] = ''
# We can probably vectorize this, but it's a small dataset so it doesn't matter
for _, row in df_semantic.iterrows():
    orig_col_name = get_original_column_name(row['orig_column_index'])
    resp = row['response']
    act_index = (df_semantic['orig_column_index'] == row['orig_column_index']) & (df_semantic['Response ID'] == row['Response ID'])
    df_semantic.loc[act_index, 'q_a_txt'] = f'Question: {orig_col_name} \n Answer: {resp}'



In [4]:
from bertopic import BERTopic

from bertopic.representation import KeyBERTInspired
# from bertopic.representation import PartOfSpeech
from bertopic.representation import MaximalMarginalRelevance

from sentence_transformers import SentenceTransformer, SimilarityFunction, util

import numpy as np

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import AgglomerativeClustering

# multi-qa-mpnet-base-dot-v1 is trained on questions and answers...
sentence_model = SentenceTransformer(
    'all-mpnet-base-v2',
    similarity_fn_name=SimilarityFunction.DOT_PRODUCT,
)
# sentence_model = SentenceTransformer('all-mpnet-base-v2')

MAX_NUM_CLUSTRERS = 10

def choose_classifier(X, max_num_clusters=MAX_NUM_CLUSTRERS):
    X1 = X / (X**2).sum(axis=-1, keepdims=True)
    vv = []
    cc = np.arange(2, len(X))
    for nclusters in cc:
        if nclusters >  max_num_clusters:
            continue
        km_model = KMeans(
            n_clusters=nclusters,
            max_iter=100,
            n_init=10,
        ).fit(X1)
        labels = km_model.labels_
        v = silhouette_score(X1, labels)
        vv.append(v)
    finish_clusters = cc[np.argmax(vv)]
    return KMeans(
        n_clusters=finish_clusters,
        max_iter=100,
        n_init=10,
    ).fit(X1)

def get_bert_classification_for_response(resp, df_bert, col_suffix, col_prefix='qa_'):
    """Gets bert classifiation for a specific response, as a dict"""
    rep_index = df_bert['Document'] == resp
    if len(df_bert[rep_index].index) != 1:
        return None
    row = df_bert[rep_index].iloc[0]
    return {
        f'{col_prefix}bert_name_qa_{col_suffix}': row['Name'],
        f'{col_prefix}bert_prob_qa_{col_suffix}': row['Probability'],
        f'{col_prefix}bert_is_repr_qa_{col_suffix}': row['Representative_document'],
        f'{col_prefix}bert_top_words_qa_{col_suffix}': row['Top_n_words'],
    }

# Create objects for using BERT
vectorizer_model = CountVectorizer(stop_words="english")
representation_model_key = KeyBERTInspired()
representation_model_mmr = MaximalMarginalRelevance(diversity=0.3)


In [5]:
df_semantic['qa_semantic_group_st'] = ''
df_semantic['semantic_cluster_id'] = ''
df_semantic['semantic_cluster_threshold'] = ''
df_semantic['qa_semantic_cluster_id'] = ''
df_semantic['qa_bert_name_plain'] = ''
df_semantic['qa_bert_prob_plain'] = ''
df_semantic['qa_bert_is_repr_plain'] = ''
df_semantic['qa_bert_top_words_plain'] = ''
df_semantic['qa_bert_name_key'] = ''
df_semantic['qa_bert_prob_key'] = ''
df_semantic['qa_bert_is_repr_key'] = ''
df_semantic['qa_bert_top_words_key'] = ''
df_semantic['qa_bert_name_key'] = ''
df_semantic['qa_bert_prob_key'] = ''
df_semantic['qa_bert_is_repr_key'] = ''
df_semantic['qa_bert_top_words_key'] = ''


for col in df_semantic['column'].unique().tolist():
    col_index = (df_semantic['column'] == col) & ~df_semantic['q_a_txt'].isnull()
    col_responses = df_semantic[col_index]['q_a_txt'].unique().tolist()
    responses = df_semantic[col_index]['response'].unique().tolist()
    # Do the sentence transformer classifications
    print(f'Make sentence transformer embeddings for {len(col_responses)} responses to {col}')
    embeddings = sentence_model.encode(col_responses, show_progress_bar=True, convert_to_numpy=True)
    classifier = choose_classifier(embeddings)
    # Make a non-numpy embeddings
    embeddings_bert = sentence_model.encode(col_responses, show_progress_bar=False)
    embeddings_resp = sentence_model.encode(responses, show_progress_bar=False)

    # This is not useful! It fails to cluster much of anything.
    do_clusters = False
    threshold = 0.75
    clusters = []
    while do_clusters:
        clusters = util.community_detection(
            embeddings_bert, 
            min_community_size=2, 
            threshold=threshold, 
        )
        if len(clusters) < 2:
             threshold = threshold - 0.05
        else:
            do_clusters = False
        if threshold < 0.05:
            do_clusters = False

    do_clusters = True
    threshold = 0.75
    resp_clusters = []
    while do_clusters:
        resp_clusters = util.community_detection(
            embeddings_resp, 
            min_community_size=4, 
            threshold=threshold, 
        )
        if len(resp_clusters) < 2:
             threshold = threshold - 0.05
        else:
            do_clusters = False
        if threshold < 0.05:
            do_clusters = False
            
    # Use different methods of BERTopic to organize responses
    topic_model_plain = BERTopic(embedding_model=sentence_model)
    topics_plain, _ = topic_model_plain.fit_transform(col_responses, embeddings_bert)
    topic_model_plain.update_topics(col_responses, vectorizer_model=vectorizer_model)
    df_bert_plain = topic_model_plain.get_document_info(col_responses)
    
    topic_model_key = BERTopic(embedding_model=sentence_model, representation_model=representation_model_key)
    topics_key, _ = topic_model_key.fit_transform(col_responses, embeddings_bert)
    topic_model_key.update_topics(col_responses, vectorizer_model=vectorizer_model)
    df_bert_key = topic_model_key.get_document_info(col_responses)

    topic_model_mmr = BERTopic(embedding_model=sentence_model, representation_model=representation_model_mmr)
    topics_mmr, _ = topic_model_mmr.fit_transform(col_responses, embeddings_bert)
    topic_model_mmr.update_topics(col_responses, vectorizer_model=vectorizer_model)
    df_bert_mmr = topic_model_mmr.get_document_info(col_responses)

    df_berts = {
        'plain': df_bert_plain,
        'key': df_bert_key,
        # 'pos': df_bert_pos,
        'mmr': df_bert_mmr,
    }
    for i, (v, resp) in enumerate(zip(embeddings, col_responses)):
        semantic_group_np = classifier.predict(v[np.newaxis])
        semantic_group_list = semantic_group_np.tolist()
        semantic_group = semantic_group_list[0]
        resp_index = col_index & (df_semantic['q_a_txt'] == resp)
        df_semantic.loc[resp_index, 'qa_semantic_group_st'] = semantic_group
        for col_suffix, df_bert in df_berts.items():
            bert_dict = get_bert_classification_for_response(resp, df_bert, col_suffix)
            if not bert_dict:
                continue
            for b_k, b_v in bert_dict.items():
                df_semantic.loc[resp_index, b_k] = b_v
    for cluster_id, cluster in enumerate(clusters):
        print(f"Cluster {cluster_id}, #{len(cluster)} responses ")
        for resp_id in cluster:
            resp = col_responses[resp_id]
            resp_index = col_index & (df_semantic['q_a_txt'] == resp)
            df_semantic.loc[resp_index, 'qa_semantic_cluster_id'] = cluster_id
    for cluster_id, cluster in enumerate(resp_clusters):
        print(f"Cluster {cluster_id}, {len(cluster)} responses, threshold: {threshold} (just the response, not q/a)")
        for resp_id in cluster:
            resp = responses[resp_id]
            resp_index = col_index & (df_semantic['response'] == resp)
            df_semantic.loc[resp_index, 'semantic_cluster_id'] = cluster_id
            df_semantic.loc[resp_index, 'semantic_cluster_threshold'] = threshold
    print('-'*50)

df_semantic.drop(columns=['q_a_txt'], inplace=True)
df_semantic.to_csv(exp_semantic_clusters_path, index=False)
print(f'Saved expanded semantic clusters to {exp_semantic_clusters_path}')

Make sentence transformer embeddings for 25 responses to Demo: Race/Ethnicity, Other, Text


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Cluster 0, 6 responses, threshold: 0.7 (just the response, not q/a)
Cluster 1, 5 responses, threshold: 0.7 (just the response, not q/a)
--------------------------------------------------
Make sentence transformer embeddings for 27 responses to Edu: Other qualifications, Text


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Cluster 0, 7 responses, threshold: 0.5499999999999998 (just the response, not q/a)
Cluster 1, 4 responses, threshold: 0.5499999999999998 (just the response, not q/a)
--------------------------------------------------
Make sentence transformer embeddings for 216 responses to Edu: Highest Degree Discipline, Text


...

Cluster 0, 4 responses, threshold: 0.49999999999999983 (just the response, not q/a)
Cluster 1, 4 responses, threshold: 0.49999999999999983 (just the response, not q/a)
--------------------------------------------------
Make sentence transformer embeddings for 115 responses to Other Thoughts: Text


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Cluster 0, 7 responses, threshold: 0.6499999999999999 (just the response, not q/a)
Cluster 1, 6 responses, threshold: 0.6499999999999999 (just the response, not q/a)
Cluster 2, 4 responses, threshold: 0.6499999999999999 (just the response, not q/a)
Cluster 3, 4 responses, threshold: 0.6499999999999999 (just the response, not q/a)
--------------------------------------------------
Make sentence transformer embeddings for 96 responses to Survey Comments: Text


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Cluster 0, 8 responses, threshold: 0.5999999999999999 (just the response, not q/a)
Cluster 1, 6 responses, threshold: 0.5999999999999999 (just the response, not q/a)
Cluster 2, 4 responses, threshold: 0.5999999999999999 (just the response, not q/a)
--------------------------------------------------
Saved expanded semantic clusters to /home/ekansa/oc-data/fair-care-semantic-clusters-expanded.csv


In [6]:
# Check to see if we have some sensible outputs.
ch_index = (df_semantic['column'] == 'Survey Comments: Text')
qa_sem_st = df_semantic[ch_index]['qa_semantic_group_st'].unique().tolist()
print(f'Example qa_semantic_group_st: {qa_sem_st}')

Example qa_semantic_group_st: [0, 1]


In [7]:
ch_index = (df_semantic['column'] == 'Survey Comments: Text')
qa_cl_ids = df_semantic[ch_index]['qa_semantic_cluster_id'].unique().tolist()
print(f'Example qa_semantic_cluster_id: {qa_cl_ids}')

Example qa_semantic_cluster_id: ['']


In [8]:
ch_index = (df_semantic['column'] == 'Survey Comments: Text')
cl_ids = df_semantic[ch_index]['semantic_cluster_id'].unique().tolist()
print(f'Example semantic_cluster_id: {cl_ids}')

Example semantic_cluster_id: ['', 0, 1, 2]
